### Ingest Auction Sales Summary CSV
This notebook ingests the auction sales report from the landing layer, standardizes the dataset, and writes the output to a bronze Delta table.

Business context:
* This file contains a summary of auction-level results rather than row-level bidding events
* It captures outcome metrics such as lots offered, lots sold, bidder counts, online activity, and sales values
* The bronze table preserves the raw summary so it can support downstream performance reporting and auction analysis

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the auction sales report path
This step assembles the full storage path for the raw auction sales summary CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific dated sales report file to ingest
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the auction summary schema
This step declares the expected structure of the CSV so Spark reads each result field with the intended type.

What happens here:
* Defines the incoming auction summary column names explicitly
* Applies consistent types for dates, counts, bidder metrics, and sales values
* Avoids schema drift that can happen with CSV inference alone


#### Step 4: Load and enrich the auction summary data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the auction sales summary file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready summary dataset
This step standardizes the DataFrame so the columns are easier to use after the bronze write.

What happens here:
* Renames source columns into a lowercase, underscore-based format
* Keeps the summary dataset consistent with downstream naming standards
* Confirms the bronze target context before persisting the table


#### Step 6: Write the bronze auction summary table
This final step persists the cleaned auction summary DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed auction results data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze summary available for downstream silver transformations and reporting


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/sales_report_*.csv'

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Define the expected structure of the auction sales summary CSV.
# Using an explicit schema prevents unwanted type inference issues for dates, counts, and sales metrics.
schema_eventitems = StructType([
        StructField('Platform', StringType(), True),
        StructField('ID', IntegerType(), True),
        StructField('Auction', StringType(), True),
        StructField('Start Date', TimestampType(), True),
        StructField('End Date', TimestampType(), True),
        StructField('Published Date', TimestampType(), True),
        StructField('Type', StringType(), True),
        StructField('Lots', IntegerType(), True),
        StructField('Lots Sold', IntegerType(), True),
        StructField('Lots with Bid', IntegerType(), True),
        StructField('Lots Sold Online', IntegerType(), True),
        StructField('Currency', StringType(),),
        StructField('Hammer Price', IntegerType(), True),
        StructField('Sold Online', DoubleType(), True),
        StructField('Absentee bids', IntegerType(), True),
        StructField('Absentee bid value', DoubleType(), True),
        StructField('Total Online Bids', IntegerType(), True),
        StructField('Registered bidders', IntegerType(), True),
        StructField('Winning Bidders', IntegerType(), True),
        StructField('Under Bidders', IntegerType(), True),
        StructField('Online Bidders', IntegerType(), True),
        StructField('Approved Bidders', IntegerType(), True),
        StructField('Page Views', IntegerType(), True),
        StructField('Clicks', StringType(), True)
])

In [ ]:
# Read the auction sales summary CSV into a Spark DataFrame using the predefined schema.
# The file is expected to contain a header row.
df_eventitems = spark.read.format('csv') \
    .option('header', 'true') \
        .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
    .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .schema(schema_eventitems) \
    .load(path)

df_eventitems = add_columns(df_eventitems)

In [ ]:
# Standardize column names so the bronze table uses lowercase, underscore-based fields.
df_eventitems = df_eventitems.withColumnsRenamed({col: col.replace(' ', '_').lower() for col in df_eventitems.columns})

In [ ]:
# Write the auction summary DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_eventitems \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.auctionsummary')